In [1]:
import os
import json

# Define the directory
directory = "test"

# List to store parsed JSON data
json_list = []

# Loop through files in the directory
for filename in os.listdir(directory):
    if filename.endswith(".json"):  # Ensure it's a JSON file
        filepath = os.path.join(directory, filename)
        with open(filepath, "r", encoding="utf-8") as file:
            data = json.load(file)  # Parse JSON into a dictionary
            json_list.append(data)

# Now, json_list contains all the parsed JSON dictionaries
test_json = json_list[0]

In [13]:
import subprocess
import json

# State is a list of field elements as previously generated.
def generate_eddsa_signature(state, nonce):
    # Convert the list of integers to a string format expected by the script
    sig_input = state + [nonce]
    try:
        result = subprocess.run(
            ["node", "generate_eddsa_signature.js", str(sig_input)],
            capture_output=True,
            text=True,
            check=True
        )
        # Parse and return the JSON output
        return json.loads(result.stdout)
    
    except subprocess.CalledProcessError as e:
        print("Error executing the script:", e)
        return None
    except json.JSONDecodeError as e:
        print("Error parsing JSON output:", e)
        return None



In [16]:
def generate_new_state_ip_list(state, new_ip):
    if new_ip in state:
        return state
    elif 0 in state:
        state[state.index(0)] = new_ip
        return state
    else:
        state = state[1:]
        state.append(new_ip)
        return state
        
    
def produce_input_with_signatures(json_input):
    old_state_signature = generate_eddsa_signature(json_input['ips'], json_input['nonce'])
    
    new_state_ips = generate_new_state_ip_list(json_input['ips'], json_input['new_ip'])
    new_state_signature = generate_eddsa_signature(new_state_ips, json_input['nonce'])
    
    json_input['initialSigR8x'] = old_state_signature['R8x']
    json_input['initialSigR8y'] = old_state_signature['R8y']
    json_input['initialSigS'] = old_state_signature['S']
    json_input['proposedSigR8x'] = new_state_signature['R8x']
    json_input['proposedSigR8y'] = new_state_signature['R8y']
    json_input['proposedSigS'] = new_state_signature['S']

In [18]:
for filename in os.listdir(directory):
        if filename.endswith(".json"):  # Ensure it's a JSON file
            filepath = os.path.join(directory, filename)
            
            # Read the JSON file
            with open(filepath, "r", encoding="utf-8") as file:
                data = json.load(file)  # Parse JSON into a dictionary
            
            # Apply the reprocessing function
            produce_input_with_signatures(data)
            
            # Write back the modified JSON
            with open(filepath, "w", encoding="utf-8") as file:
                json.dump(data, file, indent=4)  # Pretty-print with indentation
